
# Julia local small-scale test: 3D Heisenberg + DMI

This notebook is a **small local test** before the C++/Euler production run.

It fully computes the model in Julia:
- 3D Heisenberg + DMI Metropolis
- periodic boundary conditions
- energy, magnetization, heat capacity, susceptibility
- nonzero-q helical order
- parallel scan using Julia `Distributed` workers
- colored 2D slice visualization without arrows
- optional simple Julia ML baseline from Euler-generated configs

All lattice sizes here are powers of 2: `L=8,16` for local tests. `L=32,64` are for Euler/C++.


In [1]:

using Distributed
target_workers = max(1, min(Sys.CPU_THREADS ÷ 2, 4))
while nworkers() < target_workers
    addprocs(1)
end

@everywhere begin
    using Random
    using Statistics
    using LinearAlgebra
end

println("Workers: ", workers())


Workers: [2, 3, 4, 5]


In [6]:
import Pkg
Pkg.add(["DataFrames","CSV","Plots","Colors","IJulia"])

   Resolving package versions...


LoadError: KeyError: key v"6.10.2+1" not found

In [7]:

# If packages are missing, run:
# import Pkg; Pkg.add(["Plots", "DataFrames", "CSV", "Colors"])

using DataFrames
using CSV
using Plots
using Colors
gr()

mkpath("../results/processed")
mkpath("../figures/julia")


LoadError: ArgumentError: Package DataFrames not found in current path.
- Run `import Pkg; Pkg.add("DataFrames")` to install the DataFrames package.

In [3]:

@everywhere begin
    const Vec3 = NTuple{3, Float64}

    dot3(a::Vec3,b::Vec3) = a[1]*b[1]+a[2]*b[2]+a[3]*b[3]
    cross3(a::Vec3,b::Vec3) = (a[2]*b[3]-a[3]*b[2], a[3]*b[1]-a[1]*b[3], a[1]*b[2]-a[2]*b[1])
    norm3(a::Vec3) = sqrt(dot3(a,a))
    normalize3(a::Vec3) = begin
        n = norm3(a)
        n == 0 ? (1.0,0.0,0.0) : (a[1]/n, a[2]/n, a[3]/n)
    end
    add3(a::Vec3,b::Vec3) = (a[1]+b[1], a[2]+b[2], a[3]+b[3])
    mul3(c::Float64,a::Vec3) = (c*a[1], c*a[2], c*a[3])

    struct SimParams
        L::Int
        T::Float64
        J::Float64
        D::Float64
        seed::Int
        therm::Int
        samples::Int
        skip::Int
        proposal::Float64
    end

    idx(x,y,z,L) = x + L*(y-1) + L*L*(z-1)
    pbc(a,L) = mod(a-1,L)+1

    function coords(i,L)
        z = div(i-1,L*L)+1
        r = i - (z-1)*L*L
        y = div(r-1,L)+1
        x = r - (y-1)*L
        return x,y,z
    end

    function random_unit(rng)
        z = 2rand(rng)-1
        phi = 2π*rand(rng)
        r = sqrt(max(0.0,1-z*z))
        return (r*cos(phi), r*sin(phi), z)
    end

    function helical_z_initial(L,J,D)
        q0 = atan(D/J)
        n = max(1, round(Int, q0*L/(2π)))
        q = 2π*n/L
        spins = Vector{Vec3}(undef,L^3)
        for z in 1:L, y in 1:L, x in 1:L
            angle = q*(z-1)
            spins[idx(x,y,z,L)] = (cos(angle), sin(angle), 0.0)
        end
        return spins
    end

    function site_energy(spins,L,x,y,z,s::Vec3,J,D)
        E = 0.0
        dirs = ((1,0,0),(0,1,0),(0,0,1),(-1,0,0),(0,-1,0),(0,0,-1))
        ehats = ((1.0,0.0,0.0),(0.0,1.0,0.0),(0.0,0.0,1.0),(-1.0,0.0,0.0),(0.0,-1.0,0.0),(0.0,0.0,-1.0))
        for k in 1:6
            dx,dy,dz = dirs[k]
            sj = spins[idx(pbc(x+dx,L), pbc(y+dy,L), pbc(z+dz,L), L)]
            E += -J*dot3(s,sj)
            E += -D*dot3(ehats[k], cross3(s,sj))
        end
        return E
    end

    function total_energy(spins,p::SimParams)
        L,J,D = p.L,p.J,p.D
        E = 0.0
        bonds = ((1,0,0,(1.0,0.0,0.0)),(0,1,0,(0.0,1.0,0.0)),(0,0,1,(0.0,0.0,1.0)))
        for z in 1:L, y in 1:L, x in 1:L
            s = spins[idx(x,y,z,L)]
            for (dx,dy,dz,ehat) in bonds
                sj = spins[idx(pbc(x+dx,L),pbc(y+dy,L),pbc(z+dz,L),L)]
                E += -J*dot3(s,sj)
                E += -D*dot3(ehat,cross3(s,sj))
            end
        end
        return E
    end

    function propose_spin(old::Vec3,rng,proposal)
        normalize3(add3(old, mul3(proposal, random_unit(rng))))
    end

    function metropolis_sweep!(spins,p::SimParams,rng)
        L = p.L
        acc = 0
        for _ in 1:L^3
            x = rand(rng,1:L); y = rand(rng,1:L); z = rand(rng,1:L)
            i = idx(x,y,z,L)
            old = spins[i]
            new = propose_spin(old,rng,p.proposal)
            dE = site_energy(spins,L,x,y,z,new,p.J,p.D) - site_energy(spins,L,x,y,z,old,p.J,p.D)
            if dE <= 0 || rand(rng) < exp(-dE/p.T)
                spins[i] = new
                acc += 1
            end
        end
        return acc/L^3
    end

    function magnetization(spins)
        mx = mean(s[1] for s in spins)
        my = mean(s[2] for s in spins)
        mz = mean(s[3] for s in spins)
        return sqrt(mx^2+my^2+mz^2)
    end

    function structure_factor_axis(spins,L,axis,n)
        q = 2π*n/L
        rex=rey=rez=0.0
        imx=imy=imz=0.0
        for i in eachindex(spins)
            x,y,z = coords(i,L)
            r = axis==1 ? x-1 : axis==2 ? y-1 : z-1
            c = cos(q*r); sp = sin(q*r)
            S = spins[i]
            rex += S[1]*c; imx += S[1]*sp
            rey += S[2]*c; imy += S[2]*sp
            rez += S[3]*c; imz += S[3]*sp
        end
        N = L^3
        return (rex^2+imx^2+rey^2+imy^2+rez^2+imz^2)/N^2
    end

    function helical_order(spins,L)
        best=-1.0; best_q=0.0; best_axis=0; best_n=0
        for axis in 1:3, n in 1:(L-1)
            v = structure_factor_axis(spins,L,axis,n)
            if v > best
                best=v; best_q=2π*n/L; best_axis=axis; best_n=n
            end
        end
        return best,best_q,best_axis,best_n
    end

    function run_single(p::SimParams)
        rng = MersenneTwister(p.seed + 10000*p.L + round(Int,1000*p.T))
        spins = helical_z_initial(p.L,p.J,p.D)
        for _ in 1:p.therm
            metropolis_sweep!(spins,p,rng)
        end
        Es=Float64[]; Ms=Float64[]; Hs=Float64[]; accs=Float64[]
        for _ in 1:p.samples
            a=0.0
            for _ in 1:p.skip
                a += metropolis_sweep!(spins,p,rng)
            end
            push!(accs,a/p.skip)
            push!(Es,total_energy(spins,p)/p.L^3)
            push!(Ms,magnetization(spins))
            h,q,axis,n = helical_order(spins,p.L)
            push!(Hs,h)
        end
        h,q,axis,n = helical_order(spins,p.L)
        return (L=p.L,T=p.T,J=p.J,D=p.D,seed=p.seed,
                E_density_mean=mean(Es),M_abs_mean=mean(Ms),
                Cv_per_spin=p.L^3*var(Es)/p.T^2,
                chi_abs=p.L^3*var(Ms)/p.T,
                helical_order_mean=mean(Hs),
                helical_susc_like=p.L^3*var(Hs)/p.T,
                acceptance_rate=mean(accs),
                q_peak_final=q,axis_peak_final=axis,n_peak_final=n,
                spins=spins)
    end
end


In [4]:

# Single small check.
p = SimParams(8, 0.8, 1.0, 1.0, 1, 60, 60, 2, 0.7)
r = run_single(p)

println("E/N = ", r.E_density_mean)
println("|M|/N = ", r.M_abs_mean)
println("helical order = ", r.helical_order_mean)
println("q_peak = ", r.q_peak_final, " ; theory q0 = ", atan(p.D/p.J))
println("acceptance = ", r.acceptance_rate)


E/N = -2.555692409497174
|M|/N = 0.054906659400033656
helical order = 0.31119499881698826
q_peak = 5.497787143782138 ; theory q0 = 0.7853981633974483
acceptance = 0.49638671875


In [5]:

# Parallel local scan: powers of 2 only.
Ls = [8, 16]      # keep 32 and 64 for C++/Euler
temperatures = collect(0.8:0.4:2.0)
seeds = [1, 2]

params = SimParams[]
for L in Ls, T in temperatures, seed in seeds
    therm = L == 8 ? 60 : 35
    samples = L == 8 ? 80 : 35
    push!(params, SimParams(L,T,1.0,1.0,seed,therm,samples,2,0.7))
end

results = pmap(run_single, params)

df = DataFrame([
    (; L=r.L, T=r.T, J=r.J, D=r.D, seed=r.seed,
       E_density_mean=r.E_density_mean, M_abs_mean=r.M_abs_mean,
       Cv_per_spin=r.Cv_per_spin, chi_abs=r.chi_abs,
       helical_order_mean=r.helical_order_mean,
       helical_susc_like=r.helical_susc_like,
       acceptance_rate=r.acceptance_rate,
       q_peak_final=r.q_peak_final,
       axis_peak_final=r.axis_peak_final,
       n_peak_final=r.n_peak_final)
    for r in results
])

CSV.write("../results/processed/julia_local_scan.csv", df)
df


LoadError: UndefVarError: `DataFrame` not defined

In [6]:

gdf = combine(groupby(df, [:L, :T]),
    :E_density_mean => mean => :E,
    :M_abs_mean => mean => :Mabs,
    :Cv_per_spin => mean => :Cv,
    :chi_abs => mean => :chi,
    :helical_order_mean => mean => :H,
    :helical_susc_like => mean => :Hfluc,
    :acceptance_rate => mean => :acceptance,
    :q_peak_final => mean => :qpeak,
)
gdf


LoadError: UndefVarError: `df` not defined

In [7]:

# Julia quick plots.
for L in unique(gdf.L)
    d = gdf[gdf.L .== L, :]
    plot(d.T, d.E, marker=:circle, label="L=$L",
         xlabel="T", ylabel="E/N", title="Julia local energy")
end
savefig("../figures/julia/julia_energy_density.png")

plt1 = plot(title="Julia local helical order", xlabel="T", ylabel="max S(q≠0)")
for L in unique(gdf.L)
    d = gdf[gdf.L .== L, :]
    plot!(plt1, d.T, d.H, marker=:circle, label="L=$L")
end
savefig(plt1, "../figures/julia/julia_helical_order.png")

plt2 = plot(title="Julia local heat capacity", xlabel="T", ylabel="Cv/N")
for L in unique(gdf.L)
    d = gdf[gdf.L .== L, :]
    plot!(plt2, d.T, d.Cv, marker=:circle, label="L=$L")
end
savefig(plt2, "../figures/julia/julia_heat_capacity.png")


LoadError: UndefVarError: `gdf` not defined


## Colored 2D slice visualization without arrows

The color map uses:
- hue = in-plane spin angle `atan(S_y,S_x)`
- brightness = out-of-plane component `S_z`

This gives a dense and clean 2D representation of helical order.


In [8]:

function hsv_to_rgb_tuple(h,s,v)
    h = mod(h,1.0)
    i = floor(Int,6h)
    f = 6h - i
    p = v*(1-s); q = v*(1-f*s); t = v*(1-(1-f)*s)
    if i == 0
        return RGB(v,t,p)
    elseif i == 1
        return RGB(q,v,p)
    elseif i == 2
        return RGB(p,v,t)
    elseif i == 3
        return RGB(p,q,v)
    elseif i == 4
        return RGB(t,p,v)
    else
        return RGB(v,p,q)
    end
end

function spin_color(S::Vec3)
    phi = atan(S[2],S[1])
    hue = mod(phi/(2π),1.0)
    sat = 0.95
    val = 0.55 + 0.45*(S[3]+1)/2
    return hsv_to_rgb_tuple(hue,sat,val)
end

function colored_slice(spins,L; plane="xz", fixed=div(L,2))
    img = Matrix{RGB{Float64}}(undef,L,L)
    for a in 1:L, b in 1:L
        if plane == "xz"
            x,y,z = a,fixed,b
        elseif plane == "xy"
            x,y,z = a,b,fixed
        elseif plane == "yz"
            x,y,z = fixed,a,b
        else
            error("plane must be xz, xy, or yz")
        end
        img[L-b+1,a] = spin_color(spins[idx(x,y,z,L)])
    end
    return img
end

low = run_single(SimParams(16,0.6,1.0,1.0,101,120,30,2,0.7))
high = run_single(SimParams(16,2.1,1.0,1.0,102,120,30,2,0.7))

img_low = colored_slice(low.spins,16,plane="xz")
img_high = colored_slice(high.spins,16,plane="xz")

p_low = plot(img_low, axis=false, ticks=false, title="Low T colored xz slice")
p_high = plot(img_high, axis=false, ticks=false, title="High T colored xz slice")
plot(p_low, p_high, layout=(1,2), size=(900,420))
savefig("../figures/julia/colored_slice_low_high.png")


LoadError: UndefVarError: `RGB` not defined


## Optional Julia ML baseline from Euler data

The main ML pipeline is in Python/PyTorch, but these cells show how Julia can load Euler
binary configurations and train a simple logistic regression baseline using helical-order features.

For raw-spin neural networks use the Python scripts in `ml/`.


In [9]:

function read_euler_config(path::AbstractString, L::Int)
    raw = reinterpret(Float32, read(path))
    expected = L^3 * 3
    @assert length(raw) == expected "wrong binary size"
    vals = Float64.(raw)
    spins = Vector{Vec3}(undef,L^3)
    k = 1
    for i in 1:L^3
        spins[i] = (vals[k], vals[k+1], vals[k+2])
        k += 3
    end
    return spins
end

# Example after Euler ML collection:
# meta = CSV.read("../results/configs_ml/metadata.csv", DataFrame)
# meta16 = meta[meta.L .== 16, :]
# path = joinpath("../results/configs_ml", meta16.filename[1])
# spins = read_euler_config(path, 16)
# h,q,axis,n = helical_order(spins,16)


read_euler_config (generic function with 1 method)

In [10]:

# Simple logistic regression from features, implemented in Julia.
# This is a lightweight baseline, not the main ML result.

σ(x) = 1/(1+exp(-x))

function train_logistic_features(X,y; η=0.1, epochs=500)
    Xb = hcat(ones(size(X,1)), X)
    w = zeros(size(Xb,2))
    for ep in 1:epochs
        p = σ.(Xb*w)
        grad = Xb'*(p-y)/length(y)
        w .-= η*grad
    end
    return w
end

function predict_logistic(X,w)
    Xb = hcat(ones(size(X,1)), X)
    return σ.(Xb*w)
end

# Example using Julia-local scan as pseudo-ML features:
# label low T <= 0.9 as helical, high T >= 1.8 as disordered
train_df = df[(df.T .<= 0.9) .| (df.T .>= 1.8), :]
y = Float64.(train_df.T .<= 0.9)
X = Matrix(train_df[:, [:E_density_mean, :helical_order_mean, :q_peak_final]])
w = train_logistic_features(X,y,η=0.5,epochs=300)

allX = Matrix(df[:, [:E_density_mean, :helical_order_mean, :q_peak_final]])
df.P_helical_julia_feature = predict_logistic(allX,w)

plot(df.T, df.P_helical_julia_feature, seriestype=:scatter,
     xlabel="T", ylabel="P_helical", title="Julia feature ML baseline",
     legend=false)
hline!([0.5], linestyle=:dash)
savefig("../figures/julia/julia_feature_ml_baseline.png")


LoadError: UndefVarError: `df` not defined